<a href="https://colab.research.google.com/github/forcolabplesA/AEGIS/blob/main/NoPropTPUWorked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
from flax import linen as nn
from flax.training import train_state
import optax
import tensorflow_datasets as tfds
import numpy as np
import tensorflow as tf

# Enable NumPy behavior for TensorFlow tensors
tf.experimental.numpy.experimental_enable_numpy_behavior()


# NoProp Denoising Block
class DenoiseBlock(nn.Module):
    features: int

    @nn.compact
    def __call__(self, x, noisy_target, noise_level):
        # Concatenate input, noisy target, and noise level
        combined = jnp.concatenate([x, noisy_target, noise_level], axis=-1)

        # Simple MLP to denoise
        x = nn.Dense(self.features)(combined)
        x = nn.relu(x)
        x = nn.Dense(self.features)(x)
        x = nn.relu(x)

        # Output layer
        denoised = nn.Dense(10)(x)  # 10 classes for MNIST
        return denoised

# Full NoProp Network
class NoPropNet(nn.Module):
    hidden_dims: list

    @nn.compact
    def __call__(self, x, targets, noise_levels):
        # Flatten input
        x = x.reshape((x.shape[0], -1))

        outputs = []
        layer_output = x  # Initialize layer_output with flattened input

        for dim in self.hidden_dims:
            # Forward through layer
            layer_output = nn.Dense(dim)(layer_output)
            layer_output = nn.relu(layer_output)

            # Denoise block for this layer
            block = DenoiseBlock(features=dim)
            denoised = block(layer_output, targets['noisy'], targets['noise_level'])
            outputs.append(denoised)

        # Final prediction (use last layer's denoised output)
        return outputs[-1]

# Add noise to targets (diffusion-style)
def add_noise_to_targets(targets, noise_level, key):
    noise = random.normal(key, targets.shape) * noise_level
    noisy_targets = targets + noise
    return noisy_targets

# Loss function (MSE between denoised output and clean target)
def compute_loss(params, state, batch, key):
    images, labels = batch

    # One-hot encode labels
    targets = jax.nn.one_hot(labels, 10)

    # Add noise to targets
    noise_level = 0.5
    noise_level_tensor = jnp.full((images.shape[0], 1), noise_level)
    noisy_targets = add_noise_to_targets(targets, noise_level_tensor, key)

    # Prepare target dict
    target_dict = {
        'noisy': noisy_targets,
        'noise_level': noise_level_tensor
    }

    # Forward pass
    # The noise_level_tensor was being passed as a separate argument in addition to being in the target_dict.
    # Removed the redundant argument.
    preds = state.apply_fn({'params': params}, images, target_dict, noise_level_tensor)


    # MSE loss between prediction and clean target
    loss = jnp.mean((preds - targets) ** 2)
    return loss

# Training step
@jit
def train_step(state, batch, key):
    loss, grads = jax.value_and_grad(compute_loss)(state.params, state, batch, key)
    state = state.apply_gradients(grads=grads)
    return state, loss

# Evaluation
@jit
def eval_step(state, batch):
    images, labels = batch
    targets = jax.nn.one_hot(labels, 10)

    # Use zero noise for evaluation
    noise_level_tensor = jnp.zeros((images.shape[0], 1))
    target_dict = {
        'noisy': targets,
        'noise_level': noise_level_tensor
    }

    # The noise_level_tensor was being passed as a separate argument in addition to being in the target_dict.
    # Removed the redundant argument.
    preds = state.apply_fn({'params': state.params}, images, target_dict, noise_level_tensor)

    accuracy = jnp.mean(jnp.argmax(preds, -1) == labels)
    return accuracy

# Main training loop
def main():
    print("🚀 Starting NoProp on TPU v5e...")

    # Load MNIST
    ds_train = tfds.load('mnist', split='train', as_supervised=True)
    ds_test = tfds.load('mnist', split='test', as_supervised=True)

    def prepare_data(x, y):
        return x.astype(np.float32) / 255.0, y

    # Convert to numpy arrays after mapping but before batching
    ds_train = ds_train.map(prepare_data).batch(256).prefetch(10)
    ds_test = ds_test.map(prepare_data).batch(256).prefetch(10)


    # Initialize model
    key = random.PRNGKey(0)
    model = NoPropNet(hidden_dims=[256, 128])

    # Dummy input for initialization
    dummy_x = jnp.ones((1, 28, 28, 1))
    dummy_targets = {
        'noisy': jnp.ones((1, 10)),
        'noise_level': jnp.ones((1, 1))
    }
    dummy_noise_levels = jnp.ones((1, 1)) # Add dummy noise_levels
    # The noise_level_tensor was being passed as a separate argument in addition to being in the target_dict.
    # Removed the redundant argument.
    params = model.init(key, dummy_x, dummy_targets, dummy_noise_levels)['params'] # Pass dummy_noise_levels

    # Create training state
    tx = optax.adam(learning_rate=0.001)
    state = train_state.TrainState.create(
        apply_fn=jit(model.apply), # Apply jit to model.apply
        params=params,
        tx=tx
    )

    print("✅ Model initialized on TPU")
    print(f"📊 Device: {jax.devices()}")

    # Training loop
    epochs = 10
    for epoch in range(epochs):
        # Train
        train_loss = []
        for batch in tfds.as_numpy(ds_train): # Iterate over numpy batches
            key, subkey = random.split(key)
            state, loss = train_step(state, batch, subkey)
            train_loss.append(loss)

        # Evaluate
        accuracies = []
        for batch in tfds.as_numpy(ds_test): # Iterate over numpy batches
            acc = eval_step(state, batch)
            accuracies.append(acc)

        print(f"Epoch {epoch+1}/{epochs} | Loss: {np.mean(train_loss):.4f} | Acc: {np.mean(accuracies):.4f}")

    print("🎉 Training complete!")

if __name__ == "__main__":
    main()

🚀 Starting NoProp on TPU v5e...
✅ Model initialized on TPU
📊 Device: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Epoch 1/10 | Loss: 0.0157 | Acc: 0.9859
Epoch 2/10 | Loss: 0.0044 | Acc: 0.9907
Epoch 3/10 | Loss: 0.0025 | Acc: 0.9911
Epoch 4/10 | Loss: 0.0017 | Acc: 0.9914
Epoch 5/10 | Loss: 0.0012 | Acc: 0.9925
Epoch 6/10 | Loss: 0.0011 | Acc: 0.9913
Epoch 7/10 | Loss: 0.0009 | Acc: 0.9925
Epoch 8/10 | Loss: 0.0008 | Acc: 0.9908
Epoch 9/10 | Loss: 0.0007 | Acc: 0.9911
Epoch 10/10 | Loss: 0.0007 | Acc: 0.9920
🎉 Training complete!


In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 855.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
